In [1]:
!pip install numpy==1.26.4
!pip install papermill==2.3.0
!pip install pandas==1.3.5
!pip install awswrangler==3.9.0
!pip install awscli==1.33.27

  Using cached numpy-1.26.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.5
    Uninstalling numpy-1.21.5:
      Successfully uninstalled numpy-1.21.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.8.0 requires numpy<1.25.0,>=1.17.3, but you have numpy 1.26.4 which is incompatible.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.26.4 which is incompatible.
  Using cached papermill-2.3.0-py3-none-any.whl (35 kB)
  Using cached ansiwrap-0.8.4-py2.py3-none-any.whl (8.5 kB)
  Using cached textwrap3-0.9.2-py2.py3-none-any.whl (12 kB)
  Using cached awswrangler-3.9.0-py3-none-any.whl (381 kB)
  Using cached packaging-24.2-py3-none-any.whl (65 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packagi

In [2]:
import os
import time
import numpy as np
import pandas as pd
import pickle as pkl
from datetime import datetime, timedelta
import pencilbox as pb
import boto3
import awswrangler as wr

In [3]:
CON_TRINO = pb.get_connection("[Warehouse] Trino")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
def write_df_as_parquet_to_s3(data, s3_path, partition_cols=None, suffix=None):
    mode = 'overwrite_partitions' if partition_cols else 'overwrite'
    #LOGGER.info(f'writing data to s3 to path {s3_path}')

    if suffix is not None:
        s3_path = f"{s3_path.strip('/')}/{suffix}"

    wr.s3.to_parquet(
        data,
        s3_path,
        dataset=True,
        mode=mode,
        partition_cols=partition_cols
    )

In [5]:
def get_next_monday():
    today = datetime.today()
    days_until_monday = (7 - today.weekday()) % 7  # Days until next Monday
    if days_until_monday == 0:
        days_until_monday = 7  # If today is Monday, get next week's Monday
    next_monday = today + timedelta(days=days_until_monday)
    return next_monday.strftime('%Y-%m-%d')

In [6]:
from datetime import date
target_start_date = "2026-06-01"
print("week's Monday:", target_start_date)

week's Monday: 2026-06-01


In [7]:
start_date = (datetime.strptime(target_start_date, '%Y-%m-%d') - timedelta(days=60)).strftime('%Y-%m-%d')
end_date = (datetime.strptime(target_start_date, '%Y-%m-%d') + timedelta(days=7)).strftime('%Y-%m-%d')
print(start_date,end_date)

2026-04-02 2026-06-08


In [8]:
outlet_ids = pd.read_parquet("s3://prod-dse-projects/store_ops/extra_data/outlet_id/")['outlet_id']
outlet_ids_str = ", ".join(map(str, outlet_ids))
len(outlet_ids)

2329

In [9]:
temp_df = pd.read_parquet("s3://prod-dse-projects/store_ops/extra_data/monthly/slas_data/")
first_week = '2026-06-01'
last_week = '2026-06-08'
first_week, last_week

('2026-06-01', '2026-06-08')

In [10]:
start_date = (datetime.strptime(first_week, '%Y-%m-%d') - timedelta(days=1)).strftime('%Y-%m-%d')
end_date = (datetime.strptime(last_week, '%Y-%m-%d')  + timedelta(days=7)).strftime('%Y-%m-%d')
start_date, end_date

('2026-05-31', '2026-06-15')

In [11]:
festival_df = pd.read_parquet("s3://prod-dse-projects/store_ops/extra_data/festival_data_2026/")
festival_df.rename(columns={'date': 'checkout_date'}, inplace=True)
festival_df['checkout_date'] = pd.to_datetime(festival_df['checkout_date'])
anomaly_dates = list(festival_df[(festival_df.run_up==1) & (festival_df.date_is_event==1)]['checkout_date'].astype(str))
# anomaly_dates = anomaly_dates+['2025-03-14']
len(anomaly_dates)

0

In [12]:
ipc_df = pd.read_parquet("s3://prod-dse-projects/store_ops/extra_data/monthly_data/ipc_delta_data/")
ipc_df.head()

,order_date,delta
0,2026-03-06,-0.05
1,2026-03-07,-0.05
2,2026-03-08,-0.05
3,2026-03-09,-0.05
4,2026-03-10,-0.05


In [13]:
ipc_df

,order_date,delta
0,2026-03-06,-0.05
1,2026-03-07,-0.05
2,2026-03-08,-0.05
3,2026-03-09,-0.05
4,2026-03-10,-0.05
5,2026-03-11,-0.05
6,2026-03-12,-0.05
7,2026-03-13,-0.05
8,2026-03-14,-0.05
9,2026-03-15,-0.05


In [14]:
forecast_query = """
WITH raw AS (
    SELECT DISTINCT
        date AS order_date,
        DATE(updated_on) AS updated_on,
        outletid AS outlet_id,
        carts AS forecast_orders,
        ipc AS ipo,
        ROW_NUMBER() OVER (PARTITION BY outletid, date ORDER BY updated_on DESC) AS rk
    FROM logistics_data_etls.cart_projections l
    WHERE date >= DATE('{start_date}')
      AND date <= DATE('{end_date}')
      AND DATE(updated_on) <= date - INTERVAL '0' DAY
      AND outletid IN ({outlet_ids_str})
)
SELECT DISTINCT
    order_date, updated_on, outlet_id, forecast_orders, ipo
FROM raw
WHERE rk = 1
ORDER BY 1
"""

In [15]:
hitoric_hour_demand_query = """

 select order_create_dt_ist as order_date, 
 substr(cast(order_schedule_ts_ist as varchar),12,2) as hour ,
 outlet_id,
sum(total_product_quantity) as hourly_orders
from dwh.fact_sales_order_details
 where order_create_dt_ist >=  DATE('{start_date}') - interval '60' day
 and  order_create_dt_ist <=  DATE('{start_date}') - interval '8' day
 and outlet_id IN ({outlet_ids_str})
 group by 1,2,3
 order by 1,2


"""

In [16]:
# hour_df = pd.read_sql(hitoric_hour_demand_query.format(start_date = start_date, end_date = end_date, outlet_ids_str = outlet_ids_str), CON_TRINO)
# hour_df['day_of_week'] = pd.to_datetime(hour_df['order_date']).dt.day_name()
# total_order_df = hour_df.groupby(['outlet_id','order_date']).agg(total_daily_order = ('hourly_orders',sum)).reset_index()
# hour_df = hour_df.merge(total_order_df)
# hour_df['hourly_share'] = np.round(hour_df['hourly_orders']/hour_df['total_daily_order'] , 4)
# hour_df = hour_df[~(hour_df.order_date.isin(anomaly_dates))] # removing anomaly dates
# hour_df = hour_df.sort_values('order_date', ascending = False)
# temp_df = hour_df.groupby(['outlet_id','hour','day_of_week']).head(4).reset_index()
# agg_df = temp_df.groupby(['outlet_id','hour','day_of_week'])['hourly_share'].median().reset_index()
# agg_df[(agg_df.outlet_id == 1024) & (agg_df.day_of_week == 'Saturday')]

In [17]:
hitoric_hour_order_query = """

 select order_create_dt_ist as order_date, 
 substr(cast(order_schedule_ts_ist as varchar),12,2) as hour ,
 outlet_id,
 count(distinct order_id) as hourly_orders
from dwh.fact_sales_order_details
 where order_create_dt_ist >=  DATE('{start_date}') - interval '60' day
 and  order_create_dt_ist <=  DATE('{start_date}') - interval '8' day
 and outlet_id IN ({outlet_ids_str})
 group by 1,2,3
 order by 1,2


"""

In [18]:
# print("getting historic hourly order share....")
# hour_df = pd.read_sql(hitoric_hour_order_query.format(start_date = start_date, end_date = end_date, outlet_ids_str = outlet_ids_str), CON_TRINO)
# hour_df['day_of_week'] = pd.to_datetime(hour_df['order_date']).dt.day_name()
# total_order_df = hour_df.groupby(['outlet_id','order_date']).agg(total_daily_order = ('hourly_orders',sum)).reset_index()
# hour_df = hour_df.merge(total_order_df)
# hour_df['hourly_share'] = np.round(hour_df['hourly_orders']/hour_df['total_daily_order'] , 4)
# hour_df = hour_df[~(hour_df.order_date.isin(anomaly_dates))] # removing anomaly dates
# hour_df = hour_df.sort_values('order_date', ascending = False)
# temp_df = hour_df.groupby(['outlet_id','hour','day_of_week']).head(4).reset_index()
# agg_df = temp_df.groupby(['outlet_id','hour','day_of_week'])['hourly_share'].median().reset_index()

In [19]:
actual_hour_query = """

 select substr(cast(order_create_dt_ist as varchar),1,10) as order_date,
 substr(cast(order_schedule_ts_ist as varchar),12,2) as hour,
 outlet_id,
 
 count(distinct order_id) as actual_orders,
 sum(total_product_quantity) as actual_total_items
 
 from dwh.fact_sales_order_details 
 where order_create_dt_ist >=  DATE('{start_date}')
 and order_create_dt_ist <= DATE('{end_date}')
 and outlet_id IN ({outlet_ids_str})

 group by 1,2,3
 order by 1,2 
 
 
 """

In [20]:
def get_forecast_demand_data():
    
    print("getting forecasting data.....")
    forecast_df = pd.read_sql(forecast_query.format(start_date = start_date, end_date = end_date, outlet_ids_str = outlet_ids_str), CON_TRINO)
    #display(forecast_df.head())
    print(f"total outlet_ids:{forecast_df.outlet_id.nunique()}")
    print(f"total date:{forecast_df.order_date.nunique()}")
    forecast_df = forecast_df.merge(ipc_df, on = ['order_date'], how = 'left')
    forecast_df['delta'] = forecast_df['delta'].fillna(0)
    
    forecast_df['forecast_total_items'] = np.round(forecast_df['forecast_orders'] * (forecast_df['ipo'] * (1+forecast_df['delta'])) )
    display(forecast_df[forecast_df.outlet_id == 4942])
    return forecast_df

def get_hostoric_hourly_share():
    
    print("getting historic hourly share....")
    hour_df = pd.read_sql(hitoric_hour_demand_query.format(start_date = start_date, end_date = end_date, outlet_ids_str = outlet_ids_str), CON_TRINO)
    hour_df['day_of_week'] = pd.to_datetime(hour_df['order_date']).dt.day_name()
    total_order_df = hour_df.groupby(['outlet_id','order_date']).agg(total_daily_order = ('hourly_orders',sum)).reset_index()
    hour_df = hour_df.merge(total_order_df)
    hour_df['hourly_share'] = np.round(hour_df['hourly_orders']/hour_df['total_daily_order'] , 4)
    hour_df = hour_df[~(hour_df.order_date.isin(anomaly_dates))] # removing anomaly dates
    hour_df = (hour_df.assign(active_hour = hour_df['hourly_orders'] > 0).groupby(['outlet_id', 'order_date'])
               .filter(lambda x: x['active_hour'].sum() > 12))
    hour_df = hour_df.sort_values('order_date', ascending = False)
    temp_df = hour_df.groupby(['outlet_id','hour','day_of_week']).head(4).reset_index()
    agg_df = temp_df.groupby(['outlet_id','hour','day_of_week'])['hourly_share'].median().reset_index()
    return agg_df


def get_hostoric_hourly_order_share():
    
    print("getting historic hourly order share....")
    hour_df = pd.read_sql(hitoric_hour_order_query.format(start_date = start_date, end_date = end_date, outlet_ids_str = outlet_ids_str), CON_TRINO)
    hour_df['day_of_week'] = pd.to_datetime(hour_df['order_date']).dt.day_name()
    total_order_df = hour_df.groupby(['outlet_id','order_date']).agg(total_daily_order = ('hourly_orders',sum)).reset_index()
    hour_df = hour_df.merge(total_order_df)
    hour_df['hourly_order_share'] = np.round(hour_df['hourly_orders']/hour_df['total_daily_order'] , 4)
    hour_df = hour_df[~(hour_df.order_date.isin(anomaly_dates))] # removing anomaly dates
    hour_df = (hour_df.assign(active_hour = hour_df['hourly_orders'] > 0).groupby(['outlet_id', 'order_date'])
               .filter(lambda x: x['active_hour'].sum() > 12))
    hour_df = hour_df.sort_values('order_date', ascending = False)
    temp_df = hour_df.groupby(['outlet_id','hour','day_of_week']).head(4).reset_index()
    agg_df = temp_df.groupby(['outlet_id','hour','day_of_week'])['hourly_order_share'].median().reset_index()
    return agg_df


def get_forecasted_hourly_share():
    
    print("getting forecasted hourly share....")
    final_df = get_forecast_demand_data()
    final_df['day_of_week'] = pd.to_datetime(final_df['order_date']).dt.day_name()
    
    agg_df = get_hostoric_hourly_share()
    final_df = final_df.merge(agg_df[['outlet_id','hour','day_of_week','hourly_share']], on = ['outlet_id','day_of_week'] , how = 'left')
    final_df['hourly_share'] = final_df['hourly_share'].fillna(0)
    final_df['forecast_hourly_item'] = np.round(final_df['forecast_total_items'] *final_df['hourly_share'])
    
    agg_df = get_hostoric_hourly_order_share()
    final_df = final_df.merge(agg_df[['outlet_id','hour','day_of_week','hourly_order_share']], on = ['outlet_id','day_of_week','hour'] , how = 'left')
    final_df['hourly_order_share'] = final_df['hourly_order_share'].fillna(0)
    final_df['forecast_hourly_orders'] = np.round(final_df['forecast_orders'] *final_df['hourly_order_share'])
    
    display(final_df.head())
    final_df = final_df.sort_values(['order_date','hour'])
    
    return final_df

def get_final_data():
    

    final_df = get_forecasted_hourly_share()
    #print(final_df.head())
    print("postprocessing...")
    final_df = final_df[['order_date','hour','outlet_id','forecast_hourly_item','forecast_hourly_orders','forecast_orders']].rename(columns = {'order_date':'checkout_date'})
    remove_outlet_list = list(set(final_df[final_df.hour.isnull()]['outlet_id']))
    final_df = final_df[~final_df.outlet_id.isin(remove_outlet_list)]
    final_df['hour'] = final_df['hour'].astype(int)
    
    sdate = datetime.strptime(start_date, '%Y-%m-%d')
    edate = datetime.strptime(end_date, '%Y-%m-%d')
    date_list = [(sdate + timedelta(days=i)).strftime('%Y-%m-%d') 
             for i in range((edate - sdate).days + 1)]
    hour_list = [i for i in range (0,24)]
    
    date_df = pd.DataFrame()
    date_df['checkout_date'] = date_list
    hour_df = pd.DataFrame()
    hour_df['hour'] = hour_list
    outletid_df = pd.DataFrame()
    outletid_df['outlet_id'] = outlet_ids
    master_df = date_df.merge(hour_df, how = 'cross')
    master_df = master_df.merge(outletid_df, how = 'cross')
    
    save_df = master_df.merge(final_df, how='left')
    save_df[['forecast_hourly_item']] = save_df[['forecast_hourly_item']].fillna(0)
    save_df[['forecast_hourly_orders']] = save_df[['forecast_hourly_orders']].fillna(0)
    save_df[['forecast_orders']] = save_df[['forecast_orders']].fillna(0)
    
    
    assert save_df.isna().sum().sum() == 0 , 'NULL value Error'
    assert save_df.shape[0] == save_df.outlet_id.nunique() *24 * (save_df.checkout_date.nunique())
    
    return save_df

def update_peak_demand(group):
    
    
    peak_7_12 = group[(group["hour"] >= 7) & (group["hour"] <= 12)].nlargest(1, "forecast_hourly_item")
    if not peak_7_12.empty:
        peak_hour_7_12 = peak_7_12["hour"].values[0]
        peak_value_7_12 = peak_7_12["forecast_hourly_item"].values[0]
        group.loc[group["hour"].between(np.maximum(peak_hour_7_12 - 1,8), peak_hour_7_12 + 1), "forecast_hourly_item"] = peak_value_7_12

    peak_17_22 = group[(group["hour"] >= 17) & (group["hour"] <= 22)].nlargest(1, "forecast_hourly_item")
    if not peak_17_22.empty:
        peak_hour_17_22 = peak_17_22["hour"].values[0]
        peak_value_17_22 = peak_17_22["forecast_hourly_item"].values[0]
        group.loc[group["hour"].between(peak_hour_17_22 - 1, np.minimum(peak_hour_17_22 + 1,21)), "forecast_hourly_item"] = peak_value_17_22

    return group

In [21]:
data = get_final_data()
print("smoothning")
#data = data.groupby(["checkout_date", "outlet_id"], group_keys=False).apply(update_peak_demand)

temp = data.groupby(['outlet_id'])['forecast_hourly_item'].sum().reset_index()
remove_list = list(temp[temp.forecast_hourly_item==0]['outlet_id'].unique())

data = data[~data.outlet_id.isin(remove_list)]

assert data.isna().sum().sum() == 0 , 'NULL value Error'
assert data.shape[0] == data.outlet_id.nunique() * 24 * data.checkout_date.nunique()

getting forecasted hourly share....
getting forecasting data.....
total outlet_ids:2329
total date:16


,order_date,updated_on,outlet_id,forecast_orders,ipo,delta,forecast_total_items
391,2026-05-31,2026-05-31,4942,1310,4.6,0.0,6026.0
3435,2026-06-01,2026-06-01,4942,1028,4.7,0.0,4832.0
5576,2026-06-02,2026-06-02,4942,1023,4.6,0.0,4706.0
7422,2026-06-03,2026-06-03,4942,1175,4.6,0.0,5405.0
10255,2026-06-04,2026-06-04,4942,1072,4.9,0.0,5253.0
12620,2026-06-05,2026-06-04,4942,1114,4.9,0.0,5459.0
15963,2026-06-06,2026-06-04,4942,1183,5.1,0.0,6033.0
18529,2026-06-07,2026-06-04,4942,1348,5.1,0.0,6875.0
18973,2026-06-08,2026-06-04,4942,1050,4.8,0.0,5040.0
21380,2026-06-09,2026-06-04,4942,1026,4.9,0.0,5027.0


getting historic hourly share....
getting historic hourly order share....


,order_date,updated_on,outlet_id,forecast_orders,ipo,delta,forecast_total_items,day_of_week,hour,hourly_share,forecast_hourly_item,hourly_order_share,forecast_hourly_orders
0,2026-05-31,2026-05-31,6131,1403,3.6,0.0,5051.0,Sunday,00,0.00500,25.0,0.00540,8.0
1,2026-05-31,2026-05-31,6131,1403,3.6,0.0,5051.0,Sunday,06,0.00165,8.0,0.00190,3.0
2,2026-05-31,2026-05-31,6131,1403,3.6,0.0,5051.0,Sunday,07,0.02420,122.0,0.01550,22.0
3,2026-05-31,2026-05-31,6131,1403,3.6,0.0,5051.0,Sunday,08,0.04945,250.0,0.04535,64.0
4,2026-05-31,2026-05-31,6131,1403,3.6,0.0,5051.0,Sunday,09,0.06970,352.0,0.06465,91.0


postprocessing...
smoothning


In [22]:
data.describe()


,hour,outlet_id,forecast_hourly_item,forecast_hourly_orders,forecast_orders
count,861312.000000,861312.000000,861312.000000,861312.000000,861312.000000
mean,11.500000,5711.801605,349.651828,74.885317,1658.361584
std,6.922191,1609.953364,283.599319,54.511244,795.988956
min,0.000000,1024.000000,0.000000,0.000000,0.000000
25%,5.750000,4687.000000,90.000000,27.000000,1239.000000
50%,11.500000,5823.000000,328.000000,74.000000,1703.000000
75%,17.250000,7137.000000,538.000000,112.000000,2159.000000
max,23.000000,8330.000000,2445.000000,390.000000,5253.000000


In [23]:
data['checkout_date'] = pd.to_datetime(data['checkout_date'])

In [24]:
date_range = pd.date_range(start=start_date, end=end_date)
dates_mid = date_range[1:-1]
df = pd.DataFrame({'checkout_date': dates_mid})
df['week'] = (df.index // 7) + 1
first_row = pd.DataFrame({'checkout_date': [date_range[0]], 'week': [1]})
last_week_num = df['week'].max() if not df.empty else 1
last_row = pd.DataFrame({'checkout_date': [date_range[-1]], 'week': [last_week_num]})
date_df = pd.concat([first_row, df, last_row], ignore_index=True)
date_df

,checkout_date,week
0,2026-05-31,1
1,2026-06-01,1
2,2026-06-02,1
3,2026-06-03,1
4,2026-06-04,1
5,2026-06-05,1
6,2026-06-06,1
7,2026-06-07,1
8,2026-06-08,2
9,2026-06-09,2


In [25]:
final_df = data.merge(date_df, on = ['checkout_date'])
final_df.shape

(861312, 7)

In [26]:
assert final_df.shape[0] == final_df.outlet_id.nunique() * final_df.checkout_date.nunique() * 24
assert final_df.isna().sum().sum() == 0

In [27]:

final_df.head()

,checkout_date,hour,outlet_id,forecast_hourly_item,forecast_hourly_orders,forecast_orders,week
0,2026-05-31,0,8199,10.0,2.0,516.0,1
1,2026-05-31,0,8212,0.0,0.0,0.0,1
2,2026-05-31,0,8218,19.0,5.0,474.0,1
3,2026-05-31,0,8235,2.0,2.0,877.0,1
4,2026-05-31,0,8258,36.0,7.0,485.0,1


In [28]:
s3_path = f's3://prod-dse-projects/store_ops/milestone/{first_week}/hourly_item/'
print(s3_path)
write_df_as_parquet_to_s3(final_df,s3_path)

s3://prod-dse-projects/store_ops/milestone/2026-06-01/hourly_item/
